# 🌾 Pan-India Agri-Marketplace: FPO Lead-Scoring Pipeline
## Data Cleaning, Feature Engineering & 100-Point Scoring Engine

This notebook executes the All-India FPO Lead-Scoring Pipeline across 1,250+ registered FPOs in 28 States & UTs.

In [ ]:
import pandas as pd
import numpy as np

# Ingest Raw All-India FPO Dataset
df_raw = pd.read_csv('../data/fpo_marketplace_raw.csv')
print(f'Ingested {len(df_raw)} FPO records across {df_raw["state"].nunique()} States/UTs.')
df_raw.head()

## ⚙️ 100-Point Composite Scoring Engine
We apply the 4-criteria rubric:
1. Export Alignment (Max 40 Pts)
2. District Yield (Max 30 Pts)
3. Member Base Scale (Max 20 Pts)
4. Technical / CBBO Support (Max 10 Pts)

In [ ]:
def calculate_fpo_lead_score(row):
    score = 0
    if row['has_npop_organic_cert'] == 1:
        score += 25
    if row['is_apeda_cluster'] == 1:
        score += 15
    yt = row['district_yield_tier']
    if yt == 'High':
        score += 30
    elif yt == 'Moderate':
        score += 15
    else:
        score += 5
    m = row['member_count']
    if m >= 500:
        score += 20
    elif m >= 200:
        score += 10
    else:
        score += 5
    if row['has_cbbo_support'] == 1:
        score += 10
    return score

def assign_tier(score):
    if score >= 85:
        return 'Tier 1: High Priority (Immediate Outreach)'
    elif score >= 70:
        return 'Tier 2: Medium Priority (Mid-Term Onboarding)'
    else:
        return 'Tier 3: Secondary Priority (Agronomic Nurturing)'

df_clean = df_raw.copy()
df_clean['composite_score'] = df_clean.apply(calculate_fpo_lead_score, axis=1)
df_clean['outreach_tier'] = df_clean['composite_score'].apply(assign_tier)

df_clean['total_cultivated_area_ha'] = (df_clean['member_count'] * df_clean['avg_holding_ha_per_member']).round(2)
df_clean['estimated_annual_production_tonnes'] = (df_clean['total_cultivated_area_ha'] * df_clean['district_yield_tonnes_per_ha']).round(2)
df_clean['estimated_gross_turnover_inr'] = (df_clean['estimated_annual_production_tonnes'] * df_clean['crop_realization_inr_per_tonne']).round(2)

df_clean = df_clean.sort_values(by='composite_score', ascending=False).reset_index(drop=True)
print(df_clean['outreach_tier'].value_counts())